In [1]:
%pip install cryptography

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib

from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature


# 1. Tao cap khoa RSA
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()


# 2. Luu khoa bi mat
with open("private_key.pem", "wb") as f:
    f.write(private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    ))


# 3. Luu khoa cong khai
with open("public_key.pem", "wb") as f:
    f.write(public_key.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    ))


# 4. Doc thong diep
with open("message.txt", "rb") as f:
    message = f.read()

print("Thong diep:", message.decode("utf-8"))


# 5. Bam SHA-256
H = hashlib.sha256(message).hexdigest()

print("SHA-256:", H)


# 6. Tao chu ky RSA-PSS + SHA-256
signature = private_key.sign(
    message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)


# 7. Luu chu ky
with open("signature.sig", "wb") as f:
    f.write(signature)

print("Da tao chu ky va luu vao signature.sig")


# 8. Doc lai khoa cong khai va chu ky
with open("public_key.pem", "rb") as f:
    verify_key = serialization.load_pem_public_key(f.read())

with open("signature.sig", "rb") as f:
    saved_signature = f.read()


# 9. Xac minh chu ky
try:
    verify_key.verify(
        saved_signature,
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("Chu ky hop le")

except InvalidSignature:
    print("Chu ky khong hop le")


# 10. Kiem tra khi thong diep bi thay doi
modified_message = message + b" Modified"

try:
    verify_key.verify(
        saved_signature,
        modified_message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("Thong diep sua: Chu ky hop le")

except InvalidSignature:
    print("Thong diep sua: Chu ky khong hop le")

Thong diep: 
SHA-256: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
Da tao chu ky va luu vao signature.sig
Chu ky hop le
Thong diep sua: Chu ky khong hop le
